# Creation

Feature creation is a part of **feature engineering** where we generate *new* features because sometimes a machine learning model couldn't capture some patterns. It adds an entirely new column while transformation swaps features.

 ##### Why is it important?

 *Imagine* comparing height and weight in a dataset

 But it would be *way* better if there is a BMI column

 So you want to add a feature that combines the two existing ones into something more meaningful!

### Main types of creation:
1. **One-Hot Encoding** 
2. **Label Encoding**
3. **Target Encoding**
4. **Bag of Words** 
5. **TF-IDP**
6. **Derived Features**
7. **Temporal Extraction**

#### Example Data

In [1]:
import pandas as pd
import numpy as np

# Our example dataset
homes = pd.DataFrame({
    'city': ['Seattle', 'Portland', 'Seattle', 'Boston'],
    'size': ['M', 'L', 'S', 'L'],
    'sqft': [1200, 1800, 950, 2200],
    'price': [450000, 350000, 380000, 520000]
})
homes

,city,size,sqft,price
0,Seattle,M,1200,450000
1,Portland,L,1800,350000
2,Seattle,S,950,380000
3,Boston,L,2200,520000


### 1. One-Hot Encoding

**What it does:** Creates a new column for each unique category, filled with 0s and 1s.

**When to use:** When categories have no natural order, like colors, cities, or animal species.

In [13]:
onehot = pd.get_dummies(homes[['city']], dtype=int)
print(pd.concat([homes[['city']], onehot], axis=1))

       city  city_Boston  city_Portland  city_Seattle
0   Seattle            0              0             1
1  Portland            0              1             0
2   Seattle            0              0             1
3    Boston            1              0             0


### 2. Label Encoding

**What it does:** Assigns each category a single number (0, 1, 2...).

**When to use:** When categories have a natural order, like shirt sizes (S < M < L) or education level (High School < Bachelors < Masters).


In [12]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
homes['size_label'] = le.fit_transform(homes['size'])
print(homes[['city', 'size', 'size_label']])

       city size  size_label
0   Seattle    M           1
1  Portland    L           0
2   Seattle    S           2
3    Boston    L           0


### 3. Target Encoding

**What it does:** Replaces each category with the average of the target variable for that category.

**When to use:** When you have many unique categories (like zip codes or city names) where one-hot encoding would create too many columns.

In [9]:
city_avg = homes.groupby('city')['price'].mean()
homes['city_target'] = homes['city'].map(city_avg)
print(homes[['city', 'price', 'city_target']])

       city   price  city_target
0   Seattle  450000     415000.0
1  Portland  350000     350000.0
2   Seattle  380000     415000.0
3    Boston  520000     520000.0


### 4. Bag of Words

**What it does:** Counts how many times each word appears in a document. Each unique word becomes a column.

**When to use:** Simple text classification tasks where word frequency matters, like spam detection.

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

reviews = [
    "great location quiet neighborhood",
    "small kitchen old appliances",
    "great view quiet street",
    "old house great location"
]

bow = CountVectorizer()
df_bow = pd.DataFrame(bow.fit_transform(reviews).toarray(), columns=bow.get_feature_names_out())
print(df_bow)

   appliances  great  house  kitchen  location  neighborhood  old  quiet  \
0           0      1      0        0         1             1    0      1   
1           1      0      0        1         0             0    1      0   
2           0      1      0        0         0             0    0      1   
3           0      1      1        0         1             0    1      0   

   small  street  view  
0      0       0     0  
1      1       0     0  
2      0       1     1  
3      0       0     0  


### 5. TF-IDF

**What it does:** Like Bag of Words, but weights words by how unique they are. Common words get lower scores, rare words get higher scores.

**When to use:** When you want to find the most distinctive words in each document, like identifying key topics in articles.

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
df_tfidf = pd.DataFrame(tfidf.fit_transform(reviews).toarray().round(2), columns=tfidf.get_feature_names_out())
print(df_tfidf)

   appliances  great  house  kitchen  location  neighborhood   old  quiet  \
0        0.00   0.39   0.00     0.00      0.48          0.61  0.00   0.48   
1        0.53   0.00   0.00     0.53      0.00          0.00  0.41   0.00   
2        0.00   0.37   0.00     0.00      0.00          0.00  0.00   0.45   
3        0.00   0.39   0.61     0.00      0.48          0.00  0.48   0.00   

   small  street  view  
0   0.00    0.00  0.00  
1   0.53    0.00  0.00  
2   0.00    0.57  0.57  
3   0.00    0.00  0.00  


### 6. Derived Features

**What it does:** Creates new columns by applying math on existing columns.

**When to use:** When combining or calculating from existing features reveals a more meaningful signal, like turning height and weight into BMI.

In [5]:
homes['price_per_sqft'] = (homes['price'] / homes['sqft']).round(2)
print(homes[['city', 'sqft', 'price', 'price_per_sqft']])

       city  sqft   price  price_per_sqft
0   Seattle  1200  450000          375.00
1  Portland  1800  350000          194.44
2   Seattle   950  380000          400.00
3    Boston  2200  520000          236.36


### 7. Temporal Extraction

**What it does:** Pulls useful pieces of information out of a date column — like day of week, month, or whether it's a weekend.

**When to use:** When your data has dates and the timing could matter, like sales patterns by day of week or seasonal trends.


In [4]:
sales = pd.DataFrame({
    'city': ['Seattle', 'Portland', 'Seattle', 'Boston'],
    'sale_date': pd.to_datetime(['2024-01-15', '2024-03-22', '2024-07-04', '2024-11-29'])
})

sales['day_of_week'] = sales['sale_date'].dt.day_name()
sales['month'] = sales['sale_date'].dt.month
sales['quarter'] = sales['sale_date'].dt.quarter
sales['is_weekend'] = sales['sale_date'].dt.dayofweek.isin([5, 6]).astype(int)
print(sales)

       city  sale_date day_of_week  month  quarter  is_weekend
0   Seattle 2024-01-15      Monday      1        1           0
1  Portland 2024-03-22      Friday      3        1           0
2   Seattle 2024-07-04    Thursday      7        3           0
3    Boston 2024-11-29      Friday     11        4           0


### Summary

| Type | What It Does | Good For |
|------|-------------|----------|
| **One-Hot Encoding** | Category → 0/1 columns | Unordered categories (city, color) |
| **Label Encoding** | Category → single number | Ordered categories (S, M, L) |
| **Target Encoding** | Category → avg of target | Many unique categories (zip codes) |
| **Bag of Words** | Text → word counts | Simple text classification |
| **TF-IDF** | Text → uniqueness-weighted scores | Finding distinctive words |
| **Derived Features** | Math on existing columns | BMI, price per sqft, ratios |
| **Temporal Extraction** | Date → day, month, quarter | Seasonal patterns, time trends |

# The End